In [57]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split

In [ ]:
# dataset_path = r'C:/_/datasets'
# directories = [os.path.join(dataset_path, d) for d in ['vertical', 'horizontal', 'front', 'skew', 'still']]

# movement_type = 'direction'
# n_types = len(directories)
# n_types

In [ ]:
dataset_path = r'C:/_/datasets'
directories = [os.path.join(dataset_path, d) for d in ['n0', 'n1', 'n2', 'n3', 'n4']]

movement_type = 'digits'
n_types = len(directories)
n_types

5

In [ ]:
acc_range = 2
gyro_range = 250

segment_length = 80
stride = 10

headers = ['Ax', 'Ay', 'Az', 'Gx', 'Gy', 'Gz']

In [61]:
def read_multiple_csv(directory):
    df = []
    for file in os.listdir(directory):
        if file.endswith('.csv'):
            path = os.path.join(directory, file)
            temp = pd.read_csv(path, header=None)
            temp.columns = headers
            df.append(temp)
    return df

def concat_df(df):
    return pd.concat(df, ignore_index=True)

def normalize(df):
    vals = {
        'Ax': {'min': -acc_range, 'max': acc_range},
        'Ay': {'min': -acc_range, 'max': acc_range},
        'Az': {'min': -acc_range, 'max': acc_range},
        'Gx': {'min': -gyro_range, 'max': gyro_range},
        'Gy': {'min': -gyro_range, 'max': gyro_range},
        'Gz': {'min': -gyro_range, 'max': gyro_range}
    }

    for col in df.columns:
        df[col] = (df[col] - vals[col]['min']) / (vals[col]['max'] - vals[col]['min'])
    
    return df

def data_segmentation(df, segment_length, stride, label):
    segments = []
    labels = []
    for i in range(0, len(df)-segment_length+1, stride):
        seg = df.iloc[i:i+segment_length].values
        segments.append(seg)
        labels.append(label)
    return np.array(segments), np.array(labels)

In [62]:
def prepare_data(directories):
    ai_data = []
    ai_labels = []  
    for i, dir in enumerate(directories):
        df = read_multiple_csv(dir)
        df = concat_df(df)
        df = normalize(df)
        df, labels = data_segmentation(df, segment_length, stride, i)
        ai_data.append(df)
        ai_labels.append(labels)

    ai_data = np.concatenate(ai_data, axis=0)
    ai_labels = np.concatenate(ai_labels, axis=0)

    ai_data, ai_labels = shuffle(ai_data, ai_labels, random_state=42)
        
    ai_data_train, ai_data_test, ai_labels_train, ai_labels_test = train_test_split(ai_data, ai_labels, test_size=0.2, random_state=42)

    ai_data_train, ai_data_test = ai_data_train.astype(np.float32), ai_data_test.astype(np.float32)

    return ai_data_train, ai_data_test, ai_labels_train, ai_labels_test

In [63]:
ai_data_train, ai_data_test, ai_labels_train, ai_labels_test = prepare_data(directories)

In [64]:
# import tensorflow as tf # type: ignore
# from tensorflow.keras.models import Sequential # type: ignore
# from tensorflow.keras.layers import Conv1D, Flatten, Dense, Dropout # type: ignore

# model = Sequential()
# model.add(Conv1D(32, kernel_size=6, activation='relu', input_shape=(segment_length, 6))) 
# model.add(Flatten())
# model.add(Dense(n_types, activation='softmax'))

# model.compile(optimizer='adam', 
#               loss='sparse_categorical_crossentropy', 
#               metrics=['accuracy'])

# model.summary()

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, Flatten, Dense, MaxPooling1D


model = Sequential([
    Conv1D(64, n_types, activation='relu', input_shape=(segment_length, 6), padding='same'),
    MaxPooling1D(2),
    #Conv1D(128, n_types, activation='relu', padding='same'),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(n_types, activation='softmax')
])


model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


c:\Users\Maria\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [67]:
history = model.fit(
    ai_data_train, ai_labels_train,
    epochs=50,         
    batch_size=64,
    validation_data=(ai_data_test, ai_labels_test)
)

Epoch 1/50
51/51 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.2057 - loss: 1.6119 - val_accuracy: 0.4404 - val_loss: 1.5128
Epoch 2/50
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4290 - loss: 1.3985 - val_accuracy: 0.5273 - val_loss: 1.0552
Epoch 3/50
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5149 - loss: 1.0463 - val_accuracy: 0.6055 - val_loss: 0.8454
Epoch 4/50
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6211 - loss: 0.8044 - val_accuracy: 0.6390 - val_loss: 0.7537
Epoch 5/50
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6404 - loss: 0.7221 - val_accuracy: 0.6799 - val_loss: 0.7127
Epoch 6/50
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6901 - loss: 0.6649 - val_accuracy: 0.7047 - val_loss: 0.6186
Epoch 7/50
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7235 - loss: 0.6135 - val_accuracy: 0.6960 - val_loss: 0.6295
Epoch 8/50
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7269 - loss: 0.5624 - val_accuracy: 0.7730 - val_loss:

In [68]:
loss, accuracy = model.evaluate(ai_data_test, ai_labels_test)
print(f"\ntest loss: {loss:.4f}")
print(f"test accuracy: {accuracy:.4f}")

26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9859 - loss: 0.0381

test loss: 0.0475
test accuracy: 0.9789


In [69]:
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

model_name = f'{movement_type}_model_{n_types}_types_{timestamp}.tflite'
model_name

'digits_model_5_types_20250626_165201.tflite'

In [70]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.experimental_enable_resource_variables = True  
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

with open(model_name, 'wb') as f:
    f.write(tflite_model)

INFO:tensorflow:Assets written to: C:\Users\Maria\AppData\Local\Temp\tmpxg1euwhd\assets


INFO:tensorflow:Assets written to: C:\Users\Maria\AppData\Local\Temp\tmpxg1euwhd\assets


Saved artifact at 'C:\Users\Maria\AppData\Local\Temp\tmpxg1euwhd'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 40, 6), dtype=tf.float32, name='keras_tensor_23')
Output Type:
  TensorSpec(shape=(None, 5), dtype=tf.float32, name=None)
Captures:
  1972445061264: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1972307316560: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1972305349328: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1972305349520: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1972305348752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1972305348368: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1972305352208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1972305359888: TensorSpec(shape=(), dtype=tf.resource, name=None)
